# 📄 RAG PDF Question Answering

A **Retrieval-Augmented Generation (RAG)** system that allows you to ask questions about your own PDF documents. The AI answers **strictly based on your documents** — no hallucination, no outside knowledge.

---

| | |
|---|---|
| **Task** | Document Question Answering |
| **Technique** | RAG (Retrieval-Augmented Generation) |
| **LLM** | GPT via OpenAI API |
| **Embedding** | text-embedding-3-small |
| **Vector Store** | FAISS |
| **Framework** | LangChain |

## 📋 Table of Contents

1. [Setup & Install](#1-setup--install)
2. [How RAG Works](#2-how-rag-works)
3. [Load PDF Documents](#3-load-pdf-documents)
4. [Split Documents into Chunks](#4-split-documents-into-chunks)
5. [Create Embeddings & Vector Store](#5-create-embeddings--vector-store)
6. [Setup Retriever](#6-setup-retriever)
7. [Build RAG Chain](#7-build-rag-chain)
8. [Ask Questions](#8-ask-questions)
9. [Results & Conclusion](#9-results--conclusion)

---
## 1. Setup & Install

In [ ]:
!pip install langchain langchain-community langchain-openai langchain-text-splitters \
             faiss-cpu openai python-dotenv pypdf tqdm -q

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found - copy .env.example to .env and add your key"
print("API key loaded successfully!")

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("All imports successful!")

---
## 2. How RAG Works

```
📁 PDF Files
    ↓
📖 Load Documents
    ↓
✂️  Split into Chunks (1200 chars, overlap 200)
    ↓
🔢 Convert to Embeddings (vectors)
    ↓
🗄️  Store in FAISS Vector Database
    ↓
❓ User asks a question
    ↓
🔍 Retrieve top-5 most relevant chunks
    ↓
🤖 GPT answers using ONLY those chunks
    ↓
✅ Accurate, cited answer
```

---
## 3. Load PDF Documents

In [ ]:
os.makedirs("data", exist_ok=True)

# PyPDFLoader returns one Document per PDF page, with `source` and `page` metadata (used for citations)
loader = DirectoryLoader(
    path="./data",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True,
    use_multithreading=True
)

docs = loader.load()
assert docs, "No PDFs found in ./data - add at least one PDF (a sample is included)"
print(f"Loaded {len(docs)} pages")
print(f"First page preview: {docs[0].page_content[:300]}...")
print(f"Metadata: {docs[0].metadata}")

---
## 4. Split Documents into Chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

splits = text_splitter.split_documents(docs)

print(f"Total chunks created: {len(splits)}")
if splits:
    print(f"\nSample chunk:")
    print("-" * 50)
    print(splits[0].page_content)
    print("-" * 50)
    print(f"Chunk length: {len(splits[0].page_content)} characters")

---
## 5. Create Embeddings & Vector Store

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=embeddings
)

print(f"Vector store created with {len(splits)} vectors")
print(f"Embedding dimensions: 1536")

# Save / reload the index so you do not pay for embeddings on every run:
# vectorstore.save_local("faiss_index")
# vectorstore = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

---
## 6. Setup Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.3}
)

test_query = "What is this document about?"
retrieved = retriever.invoke(test_query)
print(f"Retrieved {len(retrieved)} relevant chunks for: '{test_query}'")
for i, doc in enumerate(retrieved):
    print(f"\n[Chunk {i+1}] Source: {Path(doc.metadata.get('source', 'unknown')).name}, page {doc.metadata.get('page', 0) + 1}")
    print(doc.page_content[:200] + "...")

---
## 7. Build RAG Chain

In [ ]:
def format_docs(docs):
    """Prefix each chunk with its source file and 1-based page so the LLM can cite them."""
    parts = []
    for d in docs:
        name = Path(d.metadata.get("source", "unknown")).name
        page = d.metadata.get("page")
        page_label = page + 1 if isinstance(page, int) else "?"
        parts.append(f"[source: {name}, page: {page_label}]\n{d.page_content}")
    return "\n\n".join(parts)


template = (
    "You are a strict, citation-focused assistant for a private knowledge base.\n"
    "RULES:\n"
    "1) Use ONLY the provided context to answer.\n"
    "2) If the answer is not clearly contained in the context, say: "
    "\"I don't know based on the provided documents.\"\n"
    "3) Do NOT use outside knowledge, guessing, or web information.\n"
    "4) Cite sources as (filename, page N) using the [source: ..., page: ...] "
    "tags shown in the context.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

prompt = ChatPromptTemplate.from_template(template=template)

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready!")

---
## 8. Ask Questions

In [ ]:
def ask(question):
    print(f"Question: {question}")
    print("-" * 60)
    answer = rag_chain.invoke(question)
    print(f"Answer: {answer}")
    print("=" * 60)
    return answer


ask("How many days of annual leave do employees get?")

In [ ]:
while True:
    question = input("\nQuestion: ").strip()
    if question.lower() in ['quit', 'exit', 'q']:
        break
    if not question:
        continue
    answer = rag_chain.invoke(question)
    print(f"Answer: {answer}")

---
## 9. Results & Conclusion

### Summary

| Component | Detail |
|---|---|
| **Document Loader** | Reads all PDFs from `./data/` recursively |
| **Chunking** | 1200 chars, 200 overlap, recursive separators |
| **Embedding** | OpenAI text-embedding-3-small (1536 dims) |
| **Vector Store** | FAISS — fast local similarity search |
| **Retrieval** | Top-5 chunks, similarity score ≥ 0.3 |
| **LLM** | GPT, temperature=0 |
| **Anti-hallucination** | Strict prompt — refuses to answer if not in context |

### References
- [LangChain Documentation](https://docs.langchain.com/)
- [FAISS by Meta AI](https://github.com/facebookresearch/faiss)
- [OpenAI Embeddings](https://platform.openai.com/docs/guides/embeddings)
- [RAG — Lewis et al., 2020](https://arxiv.org/abs/2005.11401)